# 연구자용 상세 분석

회사 담당자가 새 Excel을 선택해 실행하려면 [06번 쉬운 실행 화면](06_261026_크랙_샤프트_실제자료검증.ipynb)을 사용합니다. 이 노트북은 입력 경로와 저장 모델을 확인하는 상세 재현용입니다.

# 새 크랙·샤프트 Excel 분석

2026-10-26 전달 형식용입니다. 먼저 아래 파일 준비 단계를 확인하고 Run → Run All Cells로 실행합니다. 품번과 원본 위치 번호별로 PCA·거리 4종·RF·XGBoost를 계산합니다. 공개본은 실행 결과가 없는 코드이며 실제 회사 파일은 별도로 준비합니다.

MASTER 식별과 최대값 0, 100·32767 채널 처리 기준은 [새 형식·GPU 안내](../docs/새검사자료_GPU_가이드.md)를 참고하세요.

In [ ]:
from pathlib import Path
import os,sys,subprocess
import matplotlib.pyplot as plt
from IPython.display import display,HTML,Image
ROOT=Path(os.environ.get('ECT_PROJECT_ROOT','/workspace'))
if not (ROOT/'notebooks').is_dir():
    ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ectlab').is_dir())
sys.path.insert(0,str(ROOT))
from ectlab.delivery import run_delivery,feature_frame
from ectlab.exploration import download_href
INPUT=ROOT/'data/processed/new_delivery'
# 운영 서버에는 기존에 변환한 입력이 아래 폴더에 있습니다.
SERVER_INPUT=ROOT/'notebooks/20261026_새자료_입력'
if SERVER_INPUT.is_dir(): INPUT=SERVER_INPUT
if not all((INPUT/(p+s)).is_file() for p in ('crack','shaft') for s in ('_raw.csv.gz','_schema.json')):
    subprocess.run([sys.executable,str(ROOT/'scripts/convert_delivery_excel.py'),
        '--crack',str(ROOT/'data/raw/261026_크랙검사 건.xlsx'),
        '--shaft',str(ROOT/'data/raw/261026_샤프트 열처리 검사 건.xlsx'),
        '--output',str(INPUT)],check=True)
print('분석 입력:',INPUT.name)


In [ ]:
with plt.ioff():
    result=run_delivery(INPUT,ROOT/'results')
print('분석 완료:',result['folder'].name)

In [ ]:
# 보수적 시험일 포화 처리와 결과 요약. 이 셀은 새 실행 결과에 반복 적용할 수 있습니다.
import numpy as np
import pandas as pd
import joblib, shutil, json, hashlib, importlib.metadata
from ectlab.delivery import feature_frame
folder=result['folder'];stats=result['tables']['04_ML_마지막날짜시험'].copy()
pred=result['tables']['05_ML_행별점수'].copy()
pred['결과']=np.where(pred['점수']>=.9,'기준시료 유사 높음',np.where(pred['점수']<=.1,'기준시료 유사 낮음','보류'))
pred['시험일_추가포화']=False
sourceframes={profile:pd.read_csv(INPUT/(profile+'_raw.csv.gz'),low_memory=False) for profile in ('crack','shaft')}
audit=[]
for modelpath in sorted((folder/'models').glob('*.joblib')):
    bundle=joblib.load(modelpath)  # 이 실행에서 직접 만든 모델만 읽습니다.
    data=sourceframes[bundle['profile']]
    group=data[data.c3.eq(bundle['line'])&data.c4.eq(bundle['product'])&data.c6.eq(bundle['position'])]
    dates=group.set_index(['source_sheet','source_row']).c0.to_dict()
    assert all(dates[(r['source_sheet'],r['source_row'])]<bundle['test_date'] for r in bundle['train_ids'])
    metadata={k:v for k,v in bundle.items() if k!='estimator'}
    metadata['versions']={k:importlib.metadata.version(k) for k in ('numpy','scikit-learn','xgboost','joblib')}
    if modelpath.stem.endswith('_XGBoost'):
        native=modelpath.with_suffix('.ubj');bundle['estimator'].save_model(native)
        metadata['native_model']=native.name
        modelpath.with_suffix('.metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
    if '100·32767' in modelpath.name:
        x,desc=feature_frame(group,'shaft');bad=np.zeros(len(group),bool)
        for f in desc:
            if f['feature'] not in bundle['features']:continue
            values=x[f['feature']].to_numpy()
            bad|=np.isclose(np.abs(values),32767,rtol=0,atol=1e-9)
            if f['axis']=='측정값':bad|=np.isclose(values,100,rtol=0,atol=1e-9)
        bad&=group.c0.eq(bundle['test_date']).to_numpy()
        badkeys=set(zip(group.loc[bad,'source_sheet'],group.loc[bad,'source_row']))
        model='XGBoost' if modelpath.stem.endswith('_XGBoost') else 'RandomForest'
        mask=pred['검사'].eq('shaft')&pred['라인'].eq(bundle['line'])&pred['위치'].eq(bundle['position'])&pred['조건'].eq('학습일100·32767채널 제외')&pred['모델'].eq(model)
        ix=pred.index[mask];flag=np.array([(pred.at[i,'source_sheet'],pred.at[i,'source_row']) in badkeys for i in ix])
        pred.loc[ix[flag],'시험일_추가포화']=True;pred.loc[ix[flag],'결과']='보류'
    audit.append({'model':modelpath.name,'training_before_test':True,'training_rows':len(bundle['train_ids'])})
for i,row in stats.iterrows():
    g=pred[pred['검사'].eq(row['검사'])&pred['라인'].eq(row['라인'])&pred['위치'].eq(row['위치'])&pred['조건'].eq(row['조건'])&pred['모델'].eq(row['모델'])&pred['구간'].eq('마지막 날짜 시험')]
    for label,prefix in [('MASTER','MASTER'),('일반 NG','NG')]:
        part=g[g['구분'].str.contains(label,regex=False)]
        for name,decision in [('높음','기준시료 유사 높음'),('보류','보류'),('낮음','기준시료 유사 낮음')]:stats.at[i,prefix+name]=int(part['결과'].eq(decision).sum())
    stats.at[i,'추가포화_MASTER보류']=int((g['구분'].str.contains('MASTER')&g['시험일_추가포화']).sum())
result['tables']['04_ML_마지막날짜시험']=stats;result['tables']['05_ML_행별점수']=pred
stats.to_csv(folder/'04_ML_마지막날짜시험.csv',index=False,encoding='utf-8-sig');pred.to_csv(folder/'05_ML_행별점수.csv',index=False,encoding='utf-8-sig')
summary=stats.groupby(['검사','조건','모델'],sort=False)[['시험MASTER','MASTER높음','MASTER보류','MASTER낮음','시험OK','OK중높음','시험일반NG','NG높음','NG보류','NG낮음','추가포화_MASTER보류']].sum().reset_index()
summary.to_csv(folder/'10_모델비교_최종요약.csv',index=False,encoding='utf-8-sig')
with plt.ioff():
    fig,ax=plt.subplots(figsize=(13,7))
    ypos=np.arange(len(summary));left=np.zeros(len(summary));den=summary['시험MASTER'].to_numpy()
    for col,label,color in [('MASTER높음','MASTER 유사 높음','#7951B0'),('MASTER보류','보류','#A5ADBA'),('MASTER낮음','MASTER 유사 낮음','#DA9B30')]:
        values=summary[col].to_numpy();width=100*values/den;ax.barh(ypos,width,left=left,label=label,color=color)
        for y,w,l,n in zip(ypos,width,left,values):
            if n:ax.text(l+w/2,y,str(int(n)),ha='center',va='center',color='white' if color=='#7951B0' else '#15253C')
        left+=width
    labels=summary.apply(lambda r:('크랙' if r['검사']=='crack' else '샤프트')+' / '+r['조건']+' / '+r['모델']+' (n='+str(int(r['시험MASTER']))+')',axis=1)
    ax.set(yticks=ypos,yticklabels=labels,xlim=(0,100),xlabel='시험 날짜 MASTER 측정행의 분류 비중 (%)',title='이전 날짜 학습 → 마지막 날짜 MASTER 확인\n실제 진성·가성 검출 정확도가 아닌 기준시료 유사성 실험')
    ax.invert_yaxis();ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),ncol=3);fig.tight_layout();fig.savefig(folder/'모델비교_최종요약.png',dpi=150,bbox_inches='tight');plt.close(fig)
warning='학습일에 발견한 100·32767 채널을 제외하더라도 시험일에 다른 채널에서 새 포화값이 나올 수 있습니다. 보수적 비교에서는 이런 행을 점수와 무관하게 보류했습니다. 해당 행은 행별 점수 파일의 시험일_추가포화 열에서 확인합니다. 32767의 물리적 의미는 장비 담당자 확인이 필요합니다. XGBoost는 Windows에서 재사용할 수 있도록 UBJ 모델과 JSON 메타데이터도 함께 저장했습니다. MASTER 반복측정 결과를 양산 진성·가성 성능으로 해석할 수 없습니다.'
display(HTML('<h2>1차 결과와 보수적 추가 확인</h2><p>'+warning+'</p>'))
display(summary);display(Image(filename=str(folder/'모델비교_최종요약.png')))
body='<html lang="ko"><meta charset="utf-8"><title>신규 자료 1차 검토 요약</title><style>body{font:17px/1.8 sans-serif;margin:30px;color:#15253c}img{max-width:100%}table{border-collapse:collapse;font-size:13px}th,td{border:1px solid #ccd4dd;padding:7px}.note{background:#fff3d6;padding:16px}</style><h1>신규 크랙·샤프트 1차 검토</h1><p>PCA, 거리 4종, Random Forest, XGBoost를 품번·위치별로 실행했습니다. 입력 수량은 01_데이터수량.csv에서 확인합니다.</p><p>크랙은 날짜·위치별 첫 검사 건을 기준시료로 가정하며 설비판정과 다르면 확인이 필요합니다. 샤프트는 파일 내 NG마스터 명시를 우선합니다. 최대값 0행은 판정 불명으로 제외하며 일반 NG의 진성·가성은 미확정입니다.</p><p class="note">'+warning+'</p><h2>마지막 날짜 MASTER 결과</h2><img src="모델비교_최종요약.png">'+summary.to_html(index=False)+'<p>높음: 점수 0.9 이상 / 낮음: 0.1 이하 / 그 외와 시험일 새 포화: 보류. 점수는 보정된 실제 불량 확률이 아닙니다.</p><p><a href="검증보고서.html">전체 표·품번별 PCA 보기</a></p>'
(folder/'1차검토_요약.html').write_text(body,encoding='utf-8')
old=(folder/'검증보고서.html').read_text(encoding='utf-8')
insert='<p style="background:#fff3d6;padding:15px"><a href="1차검토_요약.html">보수적 추가 포화 보류를 적용한 최종 요약 보기</a><br>'+warning+'</p>'
old=old.replace('<h1>신규 크랙·샤프트 자료 검증</h1>','<h1>신규 크랙·샤프트 자료 검증</h1>'+insert)
# Replace the initial model table with the reconciled final counts.
start=old.index('<h2>04_ML_마지막날짜시험</h2>');end=old.index('<h2>',start+5)
old=old[:start]+'<h2>04_ML_마지막날짜시험</h2>'+stats.to_html(index=False)+old[end:]
(folder/'검증보고서.html').write_text(old,encoding='utf-8')
(folder/'모델_검증이력.json').write_text(json.dumps({'models':audit,'models_checked':len(audit),'test_saturation_abstention':warning},ensure_ascii=False,indent=2),encoding='utf-8')
if (INPUT/'new_delivery_final_cell.py').is_file():shutil.copyfile(INPUT/'new_delivery_final_cell.py',folder/'추가포화_최종정리.py')
result['zip']=Path(shutil.make_archive(str(folder),'zip',root_dir=folder))
print('최종 수량과 학습일 분리 검증 완료:',len(audit),'개 모델')


In [ ]:
display(HTML('<h2>수량과 MASTER 식별 확인</h2>'))
display(result['tables']['01_데이터수량'])
display(result['tables']['02_MASTER가정_확인표'])
for note in result['notes']: print('확인:',note)

In [ ]:
display(HTML('<h2>품번·위치별 PCA</h2><p>일반 NG는 미확정입니다. 샤프트는 전체 측정값, 100값 주파수 채널 전체 제외, X/Y만 사용, 100·32767 채널 전체 제외를 비교합니다.</p>'))
for name,fig in result['figures'].items():
    if name.endswith('_PCA'):
        display(HTML('<h3>'+name+'</h3>'))
        display(Image(filename=str(result['folder']/(name+'.png'))))
        plt.close(fig)

In [ ]:
display(HTML('<h2>이전 날짜 학습 → 마지막 날짜 시험</h2><p>시험 MASTER는 소수이며 반복 측정일 수 있습니다. 점수는 실제 불량 확률이 아닙니다. 일반 NG의 진성·가성 정확도는 계산할 수 없습니다.</p>'))
display(result['tables']['04_ML_마지막날짜시험'])
for name,fig in result['figures'].items():
    if not name.endswith('_PCA'):
        display(Image(filename=str(result['folder']/(name+'.png'))))
        plt.close(fig)
display(HTML('<h3>거리 4종: 진성·가성 확정 없이 참고 거리 제공</h3>'))
display(result['tables']['06_거리4종_행별점수'].groupby(['검사','라인','위치','조건','기법','해석']).size().rename('행수').reset_index())

In [ ]:
href=download_href(result['zip'],ROOT)
display(HTML(f'<h2>결과 저장 완료</h2><p><a href="{href}" download>신규 자료 검증 결과 ZIP 다운로드</a></p><p>압축을 풀고 검증보고서.html을 엽니다. PCA 그림·행별 점수·학습 모델·원본 해시·재현 코드가 함께 저장됩니다.</p>'))
print('RESULT_ZIP='+str(result['zip']))